# 📖 Bible Reading Progress Tracker —Word2Vec + CRF NER Baseline

**Purpose**  
Build a Word2Vec + CRF NER Baseline before moving to IndoBERT.
CRF is interpretable — inspect exactly which features drive predictions

| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup** | Import, paths |
| 2 | **Data** | Load CoNLL, split |
| 3 | **Word2Vec** | Train embeddings on ConLL tokens |
| 4 | **CRF** | Feature engineering (token + Word2Vec) + train + eval |
| 5 | **Inspection** | Top features, transitions, error analysis |
| 6 | **Inference** | Sanity-check pipeline on test sentences |

---
## 1. Setup

In [1]:
import re
import ast
import emoji
import sys
import pickle
import unicodedata
import warnings
import numpy as np
from pathlib import Path
from typing import Any

import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score
from gensim.models import Word2Vec

warnings.filterwarnings('ignore')
sys.path.append(str(Path('..').resolve() / 'src'))

from config.settings import MODEL_DIR, PROCESSED_DIR
from extraction.crf_extractor import CRFBibleReferenceExtractor

DATA_PATH =PROCESSED_DIR / 'NER_tasks/ner_tasks_3000.conll'
W2V_PATH  = MODEL_DIR / 'word2vec-bible.model'
SAVE_DIR  = MODEL_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

Setup complete.


## 2. Data

### 2.1 Load CoNLL

CoNLL format - one token per line, blank lines seperate sentences.

```
Efesus  B-BIBLE_REF
1       I-BIBLE_REF
-       I-BIBLE_REF
2       I-BIBLE_REF
done    O
```

In [2]:
def read_conll(filepath: Path):
    """
    Parse a CoNLL-format file into (sentences, labels) lists.
    """
    sentences, labels, raw_texts = [], [], []
    tokens, ner_tags = [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)
                    raw_texts.append(' '.join(tokens))
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if parts[0] == "-DOCSTART-":
                    continue

                tokens.append(parts[0])
                ner_tags.append(parts[-1])
    if tokens:
        sentences.append(tokens)
        labels.append(ner_tags)
        raw_texts.append(' '.join(tokens))

    return sentences, labels, raw_texts

In [3]:
CROSS_CHAPTER_VERSE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+:\d+',
    re.IGNORECASE,
)
VERSE_RANGE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+(?!\s*:)'
    r'|\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+(?!\s*:)',
    re.IGNORECASE,
)
SINGLE_VERSE_RE = re.compile(r'\d+:\d+')
CROSS_BOOK_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*[A-Za-z]'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+[A-Za-z]',
    re.IGNORECASE,
)
CHAPTER_RANGE_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*\d+'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+\d+',
    re.IGNORECASE,
)
MARKDOWN_RE = re.compile(r'[_*]')


def is_noisy(msg: str) -> bool:
    has_emoji = any(unicodedata.category(c) in ('So', 'Sm') or c in emoji.EMOJI_DATA for c in msg)
    has_markdown = bool(MARKDOWN_RE.search(msg))
    return has_emoji or has_markdown


def parse_spans(spans: Any) -> list[dict]:
    if isinstance(spans, str):
        return ast.literal_eval(spans)
    return spans or []

def detect_pattern_from_conll(tokens: list[str], ner_tags: list[str]) -> str:
    """Infer pattern pool from token list + BIO labels."""
    text     = ' '.join(tokens)
    ref_toks = [t for t, l in zip(tokens, ner_tags) if l != 'O']
    ref_text = ' '.join(ref_toks)

    if is_noisy(text):
        return 'noisy'

    if not ref_toks:
        return 'none'

    # Count distinct B- spans → multi if ≥ 2
    b_count = sum(1 for l in ner_tags if l == 'B-BIBLE_REF')
    if b_count >= 2:
        return 'multi'

    # Classify the single span by its text
    if CROSS_CHAPTER_VERSE_RE.search(ref_text):
        return 'cross_chapter_verse'
    if VERSE_RANGE_RE.search(ref_text):
        return 'verse_range'
    if SINGLE_VERSE_RE.search(ref_text):
        return 'single_verse'
    if CROSS_BOOK_RE.search(ref_text):
        return 'cross_book'
    if CHAPTER_RANGE_RE.search(ref_text):
        return 'chapter_range'
    if re.search(r'\d', ref_text):
        return 'single_chapter'
    return 'book_only'

In [4]:
sentences, labels, raw_texts = read_conll(DATA_PATH)

pattern_labels = [
    detect_pattern_from_conll(toks, tags)
    for toks, tags in zip(sentences, labels)
]

from collections import Counter
print("Pattern distribution in dataset:")
for pat, n in sorted(Counter(pattern_labels).items()):
    print(f"  {pat:22s}: {n}")

Pattern distribution in dataset:
  book_only             : 128
  chapter_range         : 740
  cross_book            : 444
  cross_chapter_verse   : 17
  multi                 : 639
  noisy                 : 678
  none                  : 106
  single_chapter        : 196
  single_verse          : 9
  verse_range           : 43


### 2.2 Train / Eval Split

80/20 stratified by label presence — ensures both splits see `B-BOOK`, `I-BOOK`, `B-CHAPTER`, `I-CHAPTER` examples.

In [5]:
(train_sent, eval_sent,
 train_labels, eval_labels,
 train_patterns, eval_patterns) = train_test_split(
    sentences, labels, pattern_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=pattern_labels,
)

print(f'Train : {len(train_sent)} | Eval : {len(eval_sent)}')
print(f'\nEval pattern breakdown:')
for pat, n in sorted(Counter(eval_patterns).items()):
    print(f'  {pat:22s}: {n}')

Train : 2400 | Eval : 600

Eval pattern breakdown:
  book_only             : 26
  chapter_range         : 148
  cross_book            : 89
  cross_chapter_verse   : 3
  multi                 : 128
  noisy                 : 136
  none                  : 21
  single_chapter        : 39
  single_verse          : 2
  verse_range           : 8


## 3. Word2Vec Embeddings

Word2Vec learns **dense vector representation of words** based on the contexts in which they appear.
Tokens that occur in similiar contexts tend to have **similar embeddings**.


These embeddings can capture relationships between tokens such as Bible book abbreviations or chapter references.

**Training setup:**
- Corpus is **lowercased** to reduce vocabulary variation
- `vector_size = 50` for a compact embedding space
- `window = 3` to capture nearby context
- `min_count = 1` since the dataset is small
- `epochs = 100` to allow sufficient training

In [6]:
token_corpus = [[t.lower() for t in sent] for sent in sentences]

w2v = Word2Vec(
    sentences=token_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=100,
)

w2v.save(str(W2V_PATH))
print(f'Word2Vec trained on {len(w2v.wv)} unique tokens,  dim=50')
print(f'Saved -> {W2V_PATH}')

for query in ['ef', 'kej', 'kor']:
    if query in w2v.wv:
        similar = w2v.wv.most_similar(query, topn=3)
        print(f'  Most similar to {query!r}: {similar}')

Word2Vec trained on 1462 unique tokens,  dim=50
Saved -> C:\one one\Desktop\bible_reading_recap_nlp\models\word2vec-bible.model
  Most similar to 'ef': [('mal', 0.8144716620445251), ('1tim', 0.7597532868385315), ('fil', 0.727959156036377)]
  Most similar to 'kej': [('kel', 0.6804136037826538), ('timotius', 0.6349866986274719), ('45-46', 0.5873854160308838)]
  Most similar to 'kor': [('rum', 0.7701948285102844), ('4️⃣', 0.7175101041793823), ('2️⃣', 0.7097870111465454)]


---
## 4. CRF

CRF models label sequences by learning:
- **Emission scores** — how likely is label X for this token's features?
- **Transition scores** — how likely is label Y to follow label X?

The key advantage over a simple classifier: it never predicts `I-BOOK` after `B-CHAPTER`.

**Features used:**
- Token shape: lowercase form, prefix/suffix, is_digit, is_title, has_hyphen
- Context: previous and next token features
- Position: beginning/end of sentence
- **Word2Vec**: cosine similarity to known BOOK and CHAPTER seed words

In [7]:
BOOK_SEEDS = [
    'kej', 'kel', 'im', 'bil', 'ul',
    'yos', 'hak', 'rut', 'sam', 'raj',
    'mat', 'mar', 'luk', 'yoh', 'kis',
    'rom', 'kor', 'gal', 'ef', 'kol'
]

CHAPTER_SEEDS = ['1','2','3','4','5','6','7','8','9']

def w2v_similarity(token: str, seeds: list) -> float:
    """Average cosine similarity between a token and a list of seed words."""
    t = token.lower()
    if t not in w2v.wv:
        return 0.0
    sims = [w2v.wv.similarity(t, s) for s in seeds if s in w2v.wv]
    return float(np.mean(sims) if sims else 0.0)

def token_features(tokens, i):
    """Feature dict for token at position i"""
    token = tokens[i]
    t = token.lower()
    
    features = {
        # Token shape
        'token.lower': t,
        'token.is_upper': token.isupper(),
        'token.is_title': token.istitle(),
        'token.is_digit': token.isdigit(),
        'token.has_hyphen': '-' in token,
        'token.has_digit': any(c.isdigit() for c in token),
        'token.prefix2': t[:2],
        'token.prefix3': t[:3],
        'token.suffix2': t[-2:],
        'token.suffix3': t[-3:],
        'token.length': len(token),
        'token.is_first': i == 0,
        'token.is_last': i == len(tokens) - 1,
        # Word2Vec Similarity
        'w2v.book_sim': round(w2v_similarity(t, BOOK_SEEDS), 2),
        'w2v.chapter_sim': round(w2v_similarity(t, CHAPTER_SEEDS), 2),
    }

    if i > 0:
        prev = tokens[i - 1]
        features.update({
            '-1:token.lower': prev.lower(),
            '-1:token.is_title': prev.istitle(),
            '-1:token.is_digit': prev.isdigit(),
            '-1:w2v.book_sim': round(w2v_similarity(prev, BOOK_SEEDS), 2),
        })
    else:
        features['BOS'] = True
    
    if i < len(tokens) - 1:
        next = tokens[i + 1]
        features.update({
            '+1:token.lower': next.lower(),
            '+1:token.is_title': next.istitle(),
            '+1:token.is_digit': next.isdigit(),
            '+1:w2v.book_sim': round(w2v_similarity(next, BOOK_SEEDS), 2),
        })
    else:
        features['EOS'] = True
    
    return features

def sent_to_features(tokens):
    return [token_features(tokens, i) for i in range(len(tokens))]

In [8]:
# Sanity check
print('Features for first token of first sentence:')
for k, v in token_features(sentences[0], 0).items():
    print(f'  {k:25s}: {v}')

Features for first token of first sentence:
  token.lower              : 14.
  token.is_upper           : False
  token.is_title           : False
  token.is_digit           : False
  token.has_hyphen         : False
  token.has_digit          : True
  token.prefix2            : 14
  token.prefix3            : 14.
  token.suffix2            : 4.
  token.suffix3            : 14.
  token.length             : 3
  token.is_first           : True
  token.is_last            : False
  w2v.book_sim             : 0.29
  w2v.chapter_sim          : 0.36
  BOS                      : True
  +1:token.lower           : inae
  +1:token.is_title        : True
  +1:token.is_digit        : False
  +1:w2v.book_sim          : 0.29


In [9]:
X_train = [sent_to_features(s) for s in train_sent]
y_train = train_labels
X_eval = [sent_to_features(s) for s in eval_sent]
y_eval = eval_labels

print(f'Train: {len(X_train)} sentences, {sum(len(x) for x in X_train)} tokens')
print(f'Eval : {len(X_eval)} sentences,  {sum(len(x) for x in X_eval)} tokens')

Train: 2400 sentences, 14307 tokens
Eval : 600 sentences,  3526 tokens


### 4.1 Train CRF

In [10]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

crf.fit(X_train, y_train)
print('CRF trained.')

CRF trained.


### 4.2 Evaluate CRF

In [11]:
y_pred_crf = crf.predict(X_eval)

print('=== CRF Evaluation ===')
print(classification_report(y_eval, y_pred_crf))
crf_f1 = f1_score(y_eval, y_pred_crf)
print(f'Overall F1: {crf_f1:.4f}')

=== CRF Evaluation ===
              precision    recall  f1-score   support

   BIBLE_REF       0.99      0.99      0.99       763

   micro avg       0.99      0.99      0.99       763
   macro avg       0.99      0.99      0.99       763
weighted avg       0.99      0.99      0.99       763

Overall F1: 0.9882


---
## 5. Inspection

CRF is fully interpretable — examine which features and transitions the model learned.

In [12]:
print('Top features per label:')
for label in ['B-BIBLE_REF', 'I-BIBLE_REF']:
    top = sorted(
        crf.state_features_.items(),
        key=lambda x: -abs(x[1])
    )
    label_features = [(f, w) for (f, l), w in top if l == label][:8]
    print(f'\n  {label}:')
    for feat, weight in label_features:
        print(f'   {feat:35s}: {weight:+.3f}')

Top features per label:

  B-BIBLE_REF:
   w2v.book_sim                       : +3.185
   -1:token.lower:20-21-1             : +2.843
   -1:w2v.book_sim                    : -2.735
   +1:token.lower:kor                 : +2.500
   -1:token.lower:done                : +2.351
   -1:token.lower:20-21-kisah         : +2.253
   +1:token.lower:-                   : -2.169
   +1:token.lower:selesai             : +2.048

  I-BIBLE_REF:
   token.is_first                     : -4.733
   -1:token.lower:-                   : +4.157
   token.has_digit                    : +3.818
   token.has_hyphen                   : +2.462
   -1:w2v.book_sim                    : +2.249
   token.suffix2:️⃣                   : -2.247
   -1:token.lower:16-                 : +1.812
   +1:token.lower:kor                 : -1.727


In [13]:
print('Top transition weights:')
top_transitions = sorted(
    crf.transition_features_.items(),
    key=lambda x: -x[1]
)[:10]
for (from_label, to_label), weight in top_transitions:
    print(f'  {from_label:12s} → {to_label:12s}: {weight:+.3f}')

Top transition weights:
  O            → O           : +4.098
  B-BIBLE_REF  → I-BIBLE_REF : +2.200
  I-BIBLE_REF  → I-BIBLE_REF : +0.766
  O            → B-BIBLE_REF : +0.726
  I-BIBLE_REF  → O           : -0.562
  B-BIBLE_REF  → O           : -1.759
  I-BIBLE_REF  → B-BIBLE_REF : -2.964
  B-BIBLE_REF  → B-BIBLE_REF : -3.215
  O            → I-BIBLE_REF : -4.680


In [14]:
errors = []
for tokens, true_seq, pred_seq in zip(eval_sent, y_eval, crf.predict(X_eval)):
    for token, true_label, pred_label in zip(tokens, true_seq, pred_seq):
        if true_label != pred_label:
            errors.append({
                'token': token,
                'true_label': true_label,
                'pred_label': pred_label,
            })

print(f'Total errors : {len(errors)}\n')
print(f'{"Token":20s} {"True":15s} {"Predicted":15s}')
print('-' * 52)
for e in errors[:20]:
    print(f'{e["token"]:20s} {e["true_label"]:15s} {e["pred_label"]:15s}')

Total errors : 16

Token                True            Predicted      
----------------------------------------------------
Jojo                 O               B-BIBLE_REF    
Yer13-16             B-BIBLE_REF     I-BIBLE_REF    
Yohanes              O               B-BIBLE_REF    
Bil                  B-BIBLE_REF     I-BIBLE_REF    
Ayub                 B-BIBLE_REF     O              
Jam                  O               B-BIBLE_REF    
7-9                  O               I-BIBLE_REF    
Luk                  I-BIBLE_REF     B-BIBLE_REF    
Micah                B-BIBLE_REF     O              
6:8                  I-BIBLE_REF     O              
-                    O               I-BIBLE_REF    
2👉🎄                  O               I-BIBLE_REF    
daniel               B-BIBLE_REF     O              
03Bambang            O               B-BIBLE_REF    
Rom                  B-BIBLE_REF     I-BIBLE_REF    
Bil31                B-BIBLE_REF     O              


In [15]:
crf_save_path = SAVE_DIR / 'crf-bible-ner.pkl'
with open(crf_save_path, 'wb') as f:
    pickle.dump(crf, f)
print(f'CRF saved → {crf_save_path}')

CRF saved → C:\one one\Desktop\bible_reading_recap_nlp\models\crf-bible-ner.pkl
